In [1]:
import numpy as np
import sounddevice as sd
# Frequencies for the 4th octave (A4 = 440 Hz)
NOTE_OFFSETS = {
    "C": -9,
    "C#": -8,
    "Db": -8,
    "D": -7,
    "D#": -6,
    "Eb": -6,
    "E": -5,
    "F": -4,
    "F#": -3,
    "Gb": -3,
    "G": -2,
    "G#": -1,
    "Ab": -1,
    "A": 0,
    "A#": 1,
    "Bb": 1,
    "B": 2,
}

#SAMPLE_RATE = 44100
SAMPLE_RATE = int(sd.query_devices(kind="output")["default_samplerate"])
print("Using sample rate:", SAMPLE_RATE)

def note_frequency(note: str, octave: int) -> float:
    """
    Compute the frequency of a note.
    Example: note_frequency("C", 4) -> 261.63 Hz
    """
    if note not in NOTE_OFFSETS:
        raise ValueError(f"Unknown note: {note}")

    semitones = NOTE_OFFSETS[note] + (octave - 4) * 12
    return 440.0 * (2 ** (semitones / 12))

Using sample rate: 48000


In [2]:
def play_note(note: str, octave: int, duration: float, volume: float):
    frequency = note_frequency(note, octave)

    sample_rate = int(sd.query_devices(kind="output")["default_samplerate"])

    samples = int(sample_rate * duration)
    t = np.arange(samples) / sample_rate

    waveform = np.sin(2 * np.pi * frequency * t)

    # Apply volume (0.0 = silent, 1.0 = maximum)
    waveform *= volume

    # Fade in/out
    fade_samples = min(int(sample_rate * 0.01), samples // 2)
    waveform[:fade_samples] *= np.linspace(0, 1, fade_samples)
    waveform[-fade_samples:] *= np.linspace(1, 0, fade_samples)

    sd.play(waveform.astype(np.float32), sample_rate)
    sd.wait()

In [3]:
play_note("D", 2, 1.0, 1.8)  # C3, 1 second, 80% volume


In [32]:
# with instrument

import fluidsynth
import time
import os

SOUNDFONT = "/home/red/Documents/Projects/Jammify/FluidR3_GM.sf2"
print("Exists:", os.path.exists(SOUNDFONT))
print("Readable:", os.access(SOUNDFONT, os.R_OK))
print("Size:", os.path.getsize(SOUNDFONT) if os.path.exists(SOUNDFONT) else None)


fs = fluidsynth.Synth()
fs.setting("synth.gain", 2.0)

fs.start(driver="pulseaudio")
#fs.start()

sfid = fs.sfload(SOUNDFONT)

if sfid == -1:
    raise RuntimeError(f"Could not load SoundFont: {SOUNDFONT}")

print("Loaded SoundFont ID:", sfid)


for program in [0, 24, 40, 56, 73]:
    name = fs.sfpreset_name(sfid, 0, program)
    print(program, name)


def note_to_midi(note, octave):
    """Convert note name and octave to MIDI note number."""
    notes = {
        "C": 0,
        "C#": 1,
        "D": 2,
        "D#": 3,
        "E": 4,
        "F": 5,
        "F#": 6,
        "G": 7,
        "G#": 8,
        "A": 9,
        "A#": 10,
        "B": 11,
    }

    return 12 * (octave + 1) + notes[note]


def play_note(midi_note, duration, volume):
    """
    Play a MIDI note.

    Args:
        midi_note: MIDI note number (C4 = 60)
        duration: seconds
        volume: 0.0 - 1.0
    """
    channel = 0
    velocity = int(volume * 127)

    fs.noteon(channel, midi_note, velocity)

    time.sleep(duration)

    fs.noteoff(channel, midi_note)


Exists: True
Readable: True
Size: 148398306
Loaded SoundFont ID: 1
0 Yamaha Grand Piano
24 Nylon String Guitar
40 Violin
56 Trumpet
73 Flute


fluidsynth: warning: SDL3 not initialized, SDL3 audio driver won't be usable. Have you called SDL_Init(SDL_INIT_AUDIO) ?
fluidsynth: Using PulseAudio driver
fluidsynth: warning: Failed to set thread to high priority
fluidsynth: warning: Failed to set thread to high priority


In [33]:
instruments = {
    # Piano
    "acoustic_grand_piano": 0,
    "bright_acoustic_piano": 1,
    "electric_grand_piano": 2,
    "honky_tonk_piano": 3,
    "electric_piano_1": 4,
    "electric_piano_2": 5,
    "harpsichord": 6,
    "clavinet": 7,

    # Chromatic Percussion
    "celesta": 8,
    "glockenspiel": 9,
    "music_box": 10,
    "vibraphone": 11,
    "marimba": 12,
    "xylophone": 13,
    "tubular_bells": 14,
    "dulcimer": 15,

    # Organ
    "drawbar_organ": 16,
    "percussive_organ": 17,
    "rock_organ": 18,
    "church_organ": 19,
    "reed_organ": 20,
    "accordion": 21,
    "harmonica": 22,
    "tango_accordion": 23,

    # Guitar
    "nylon_string_guitar": 24,
    "steel_string_guitar": 25,
    "jazz_guitar": 26,
    "clean_electric_guitar": 27,
    "muted_electric_guitar": 28,
    "overdriven_guitar": 29,
    "distortion_guitar": 30,
    "guitar_harmonics": 31,

    # Bass
    "acoustic_bass": 32,
    "finger_bass": 33,
    "pick_bass": 34,
    "fretless_bass": 35,
    "slap_bass_1": 36,
    "slap_bass_2": 37,
    "synth_bass_1": 38,
    "synth_bass_2": 39,

    # Strings
    "violin": 40,
    "viola": 41,
    "cello": 42,
    "contrabass": 43,
    "tremolo_strings": 44,
    "pizzicato_strings": 45,
    "orchestral_harp": 46,
    "timpani": 47,

    # Ensemble
    "string_ensemble_1": 48,
    "string_ensemble_2": 49,
    "synth_strings_1": 50,
    "synth_strings_2": 51,
    "choir_aahs": 52,
    "voice_oohs": 53,
    "synth_voice": 54,
    "orchestra_hit": 55,

    # Brass
    "trumpet": 56,
    "trombone": 57,
    "tuba": 58,
    "muted_trumpet": 59,
    "french_horn": 60,
    "brass_section": 61,
    "synth_brass_1": 62,
    "synth_brass_2": 63,

    # Reed
    "soprano_sax": 64,
    "alto_sax": 65,
    "tenor_sax": 66,
    "baritone_sax": 67,
    "oboe": 68,
    "english_horn": 69,
    "bassoon": 70,
    "clarinet": 71,

    # Pipe
    "piccolo": 72,
    "flute": 73,
    "recorder": 74,
    "pan_flute": 75,
    "blown_bottle": 76,
    "shakuhachi": 77,
    "whistle": 78,
    "ocarina": 79,

    # Synth Lead
    "lead_square": 80,
    "lead_sawtooth": 81,
    "lead_calliope": 82,
    "lead_chiff": 83,
    "lead_charang": 84,
    "lead_voice": 85,
    "lead_fifths": 86,
    "lead_bass_lead": 87,

    # Synth Pad
    "pad_new_age": 88,
    "pad_warm": 89,
    "pad_poly_synth": 90,
    "pad_choir": 91,
    "pad_bowed": 92,
    "pad_metallic": 93,
    "pad_halo": 94,
    "pad_sweep": 95,

    # Synth Effects
    "fx_rain": 96,
    "fx_soundtrack": 97,
    "fx_crystal": 98,
    "fx_atmosphere": 99,
    "fx_brightness": 100,
    "fx_goblins": 101,
    "fx_echoes": 102,
    "fx_scifi": 103,

    # Ethnic
    "sitar": 104,
    "banjo": 105,
    "shamisen": 106,
    "koto": 107,
    "kalimba": 108,
    "bagpipe": 109,
    "fiddle": 110,
    "shanai": 111,

    # Percussive
    "tinkle_bell": 112,
    "agogo": 113,
    "steel_drums": 114,
    "woodblock": 115,
    "taiko_drum": 116,
    "melodic_tom": 117,
    "synth_drum": 118,
    "reverse_cymbal": 119,

    # Sound Effects
    "guitar_fret_noise": 120,
    "breath_noise": 121,
    "seashore": 122,
    "bird_tweet": 123,
    "telephone_ring": 124,
    "helicopter": 125,
    "applause": 126,
    "gunshot": 127,
}

In [34]:
def play_instrument(note, octave, duration, volume, instrument):
    """
    Select an instrument and play a note.
    """
    

    if instrument not in instruments:
        raise ValueError(f"Unknown instrument: {instrument}")

    channel = 0

    # Select General MIDI instrument
    fs.program_select(
        channel,
        sfid,
        0,
        instruments[instrument]
    )

    midi_note = note_to_midi(note, octave)

    # Delegate actual playing
    play_note(midi_note, duration, volume)

In [47]:
play_instrument("C", 2, 1.0, 1.0, "finger_bass")

In [39]:
from mingus.core import chords


def chord_to_notes(chord_name, octave=4):
    """
    Convert chord name to note names.
    Supports inversions like C/E.
    """

    if "/" in chord_name:
        base, bass = chord_name.split("/")
        notes = chords.from_shorthand(base)

        # Put the bass note first
        if bass in notes:
            notes.remove(bass)
        notes.insert(0, bass)

    else:
        notes = chords.from_shorthand(chord_name)

    return [(note, octave) for note in notes]


print(chord_to_notes("C", octave=2))
print(chord_to_notes("C/G"))


[('C', 2), ('E', 2), ('G', 2)]
[('G', 4), ('C', 4), ('E', 4)]


In [72]:
import threading
import time

def play_chord(chord_name, octave, duration, volume, instrument, wait = 0):
    notes = chord_to_notes(chord_name, octave)

    threads = []

    for note, octave in notes:
        t = threading.Thread(
            target=play_instrument,
            args=(note, octave, duration, volume, instrument)
        )

        threads.append(t)
        t.start()

        time.sleep(wait)
    for t in threads:
        t.join()


In [80]:
play_chord("C", 3, 1.0, 0.5, "clean_electric_guitar", wait=0)
play_chord("C", 4, 1.0, 0.5, "overdriven_guitar", wait = 0.5)
play_chord("C", 5, 1.0, 0.5, "acoustic_grand_piano", wait = 0.3)
